# Multi-source projections framework

Goal: stop treating "the projection" as one number from one place. Instead, keep a
tidy table where **every source contributes rows in the same shape**, so adding a
new source later (ESPN, a homebrew model, FantasyPros, whatever) means writing one
small adapter and nothing else changes.

Why this is worth doing before more modeling: consensus/ensemble projections
reliably beat any single source, and it's cheap. It also makes disagreement
visible -- when ESPN says 210 and Sleeper says 260, that gap is *information*
(usually a role or injury question), and a dashboard that surfaces it is telling
you something no single-source app can.

This notebook: (1) inventories what data each source actually gives us, (2) defines
the common schema, (3) writes working adapters, (4) builds a consensus, (5) saves
`data/projections_all.parquet`.

Run from the repo root.

In [ ]:
import os
import requests
import polars as pl
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
def find_repo_root(marker="data", start=None):
    """Walk up from `start` (default: cwd) until a directory containing `marker` is found."""
    path = Path(start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(f"Could not locate a '{marker}' directory above {path}")

DATA = find_repo_root() / "data"

SEASON = int(os.environ.get("SEASON", 2026))
print("season:", SEASON)

season: 2026


## 1. What is actually in Sleeper's payload?

We currently keep only the three point totals, but the `stats` dict may carry
**component** projections (yards, receptions, TDs). If it does, we get those for
free from a call we already make -- and they're the raw material for any future
component-level model. Let's find out rather than assume.

In [6]:
PROJ_BASE = "https://api.sleeper.com/projections/nfl"
POSITIONS = ["QB", "RB", "WR", "TE", "K", "DEF"]

params = [("season_type", "regular"), ("order_by", "pts_ppr")]
params += [("position[]", p) for p in POSITIONS]
raw = requests.get(f"{PROJ_BASE}/{SEASON}", params=params, timeout=30).json()
print("records:", len(raw))

# Which stat keys appear, and how often? Frequency matters -- a key present on
# 5% of players isn't something to build on.
from collections import Counter
counter = Counter()
for row in raw:
    counter.update((row.get("stats") or {}).keys())

freq = pl.DataFrame(
    {"stat": list(counter.keys()), "n_players": list(counter.values())}
).with_columns(pct=(pl.col("n_players") / len(raw) * 100).round(1)).sort("n_players", descending=True)
freq.head(40)

records: 3292


stat,n_players,pct
str,i64,f64
"""adp_2qb""",3292,100.0
"""adp_dynasty""",3292,100.0
"""adp_dynasty_2qb""",3292,100.0
"""adp_dynasty_half_ppr""",3292,100.0
"""adp_dynasty_ppr""",3292,100.0
…,…,…
"""cmp_pct""",76,2.3
"""pass_cmp""",76,2.3
"""pass_fd""",76,2.3


Scan that list. Point totals (`pts_*`) should be near-universal. If you also see
things like passing/rushing/receiving yards, receptions, and TD keys on a healthy
share of players, then component-level modeling is viable later with **no new data
source** -- we'd just widen what `fetch_projections.py` keeps.

## 2. The common schema

Every source emits rows in exactly this shape:

| column | meaning |
|---|---|
| `source` | who produced it: `sleeper`, `espn`, `model`, ... |
| `player_id` | Sleeper player id -- our universal key (the crosswalk maps others to it) |
| `name`, `position`, `team` | identity, for display |
| `horizon` | `season` or `week` |
| `week` | week number, or null for season |
| `scoring` | `ppr`, `half_ppr`, `std` -- **critical**: never average across different scoring |
| `points` | the projected points |

Long format (one row per source x player) rather than wide (a column per source) is
the right call: adding a source adds *rows*, not schema changes, so nothing
downstream breaks.

In [7]:
SCHEMA_COLS = ["source", "player_id", "name", "position", "team",
               "horizon", "week", "scoring", "points"]

def conform(df, source):
    """Force a frame into the standard schema so every source is interchangeable."""
    out = df.with_columns(pl.lit(source).alias("source"))
    for c in SCHEMA_COLS:
        if c not in out.columns:
            out = out.with_columns(pl.lit(None).alias(c))
    return out.select(SCHEMA_COLS).with_columns(
        pl.col("player_id").cast(pl.Utf8),
        pl.col("points").cast(pl.Float64),
    )

## 3. Adapter: Sleeper

We already have this data on disk from `fetch_projections.py` -- we just reshape it
from wide (three point columns) to long (one row per scoring system).

In [8]:
season_wide = pl.read_parquet(DATA / "projections_season.parquet")

scoring_map = {"pts_ppr": "ppr", "pts_half_ppr": "half_ppr", "pts_std": "std"}
frames = []
for col, label in scoring_map.items():
    frames.append(
        season_wide
        .select(["player_id", "name", "position", "team", pl.col(col).alias("points")])
        .with_columns(
            horizon=pl.lit("season"),
            week=pl.lit(None, dtype=pl.Int64),
            scoring=pl.lit(label),
        )
        .drop_nulls("points")
    )
sleeper_long = conform(pl.concat(frames), "sleeper")
print("sleeper rows:", sleeper_long.height)
sleeper_long.head()

sleeper rows: 1911


source,player_id,name,position,team,horizon,week,scoring,points
str,str,str,str,str,str,i64,str,f64
"""sleeper""","""4984""","""Josh Allen""","""QB""","""BUF""","""season""",null,"""ppr""",361.5
"""sleeper""","""9509""","""Bijan Robinson""","""RB""","""ATL""","""season""",null,"""ppr""",324.9
"""sleeper""","""11564""","""Drake Maye""","""QB""","""NEP""","""season""",null,"""ppr""",320.76
"""sleeper""","""4881""","""Lamar Jackson""","""QB""","""BAL""","""season""",null,"""ppr""",320.0
"""sleeper""","""11566""","""Jayden Daniels""","""QB""","""WAS""","""season""",null,"""ppr""",313.72


## 4. Adapter: ESPN

ESPN publishes its own season projections. In a league that hasn't drafted, every
player is a free agent, so `free_agents()` is the way to reach the whole pool
preseason.

Two honest caveats: ESPN's projections use **ESPN's scoring**, which is not
necessarily your league's -- we label it `espn_default` rather than pretending it's
PPR. And preseason availability varies; if this returns nothing, that's expected
this early and the cell fails soft.

In [9]:
espn_long = None
try:
    from espn_api.football import League

    league = League(
        league_id=int(os.environ["LEAGUE_ID"]), year=SEASON,
        espn_s2=os.environ["ESPN_S2"], swid=os.environ["SWID"],
    )
    fa = league.free_agents(size=600)
    print("free agents returned:", len(fa))

    rows = [{
        "espn_id": p.playerId,
        "name": p.name,
        "position": p.position,
        "team": p.proTeam,
        "points": getattr(p, "projected_total_points", None),
    } for p in fa]
    espn_raw = pl.DataFrame(rows).drop_nulls("points").filter(pl.col("points") > 0)
    print("with projections:", espn_raw.height)
    espn_raw.head()
except Exception as e:
    print(f"ESPN projections unavailable ({e.__class__.__name__}: {e})")
    print("Expected this far out from the season -- rerun closer to the draft.")

ESPN projections unavailable (ModuleNotFoundError: No module named 'espn_api')
Expected this far out from the season -- rerun closer to the draft.


ESPN keys on its own player ids, so we map to Sleeper ids through the crosswalk we
already built. That crosswalk was worth every minute -- this is the second time it's
saved us from name-matching.

In [10]:
if 'espn_raw' in dir() and espn_raw is not None and espn_raw.height:
    crosswalk = (
        pl.read_parquet(DATA / "playerids.parquet")
        .select(["sleeper_id", "espn_id"])
        .with_columns(
            pl.col("sleeper_id").cast(pl.Utf8),
            pl.col("espn_id").cast(pl.Int64),
        )
        .drop_nulls()
    )
    joined = (
        espn_raw.with_columns(pl.col("espn_id").cast(pl.Int64))
        .join(crosswalk, on="espn_id", how="inner")
        .rename({"sleeper_id": "player_id"})
        .with_columns(
            horizon=pl.lit("season"),
            week=pl.lit(None, dtype=pl.Int64),
            scoring=pl.lit("espn_default"),
        )
    )
    espn_long = conform(joined, "espn")
    matched = espn_long.height
    print(f"matched {matched} of {espn_raw.height} ESPN players to Sleeper ids")
    espn_long.head()
else:
    print("Skipping ESPN mapping -- no ESPN projections loaded.")

Skipping ESPN mapping -- no ESPN projections loaded.


## 5. Combine, and build a consensus

The consensus rule that matters: **only average within the same scoring system.**
Averaging a PPR number with an ESPN-default number produces a figure that means
nothing. So we group by (player, horizon, week, scoring).

Right now that means ESPN won't blend with Sleeper until we convert ESPN's scoring
(a future adapter improvement). It still earns its place in the table as a
comparison column -- disagreement between sources is itself signal.

In [11]:
parts = [sleeper_long] + ([espn_long] if espn_long is not None else [])
all_proj = pl.concat(parts)

DATA.mkdir(exist_ok=True)
all_proj.write_parquet(DATA / "projections_all.parquet")
print("wrote data/projections_all.parquet |", all_proj.height, "rows")

print(all_proj.group_by(["source", "scoring"]).agg(n=pl.len()).sort("source"))

wrote data/projections_all.parquet | 1911 rows
shape: (3, 3)
┌─────────┬──────────┬─────┐
│ source  ┆ scoring  ┆ n   │
│ ---     ┆ ---      ┆ --- │
│ str     ┆ str      ┆ u32 │
╞═════════╪══════════╪═════╡
│ sleeper ┆ std      ┆ 637 │
│ sleeper ┆ ppr      ┆ 637 │
│ sleeper ┆ half_ppr ┆ 637 │
└─────────┴──────────┴─────┘


In [12]:
# Consensus: mean across sources within an identical scoring system.
consensus = (
    all_proj
    .group_by(["player_id", "name", "position", "team", "horizon", "week", "scoring"])
    .agg(
        points=pl.col("points").mean(),
        n_sources=pl.col("source").n_unique(),
        spread=(pl.col("points").max() - pl.col("points").min()),
    )
    .sort("points", descending=True)
)

# Where sources disagree most -- often a role or injury question worth a look.
print("Biggest disagreements:")
consensus.filter(pl.col("n_sources") > 1).sort("spread", descending=True).head(10)

Biggest disagreements:


player_id,name,position,team,horizon,week,scoring,points,n_sources,spread
str,str,str,str,str,i64,str,f64,u32,f64


## 6. Adding a source later

The whole point of the schema is that a new source is a small, self-contained job:

1. Fetch it however it's fetched.
2. Map its player ids to Sleeper ids via `playerids.parquet` (or emit Sleeper ids directly, as a homebrew model would).
3. Label its `scoring` honestly -- convert to a shared system if you want it in the consensus.
4. `conform(df, "your_source")` and concat.

A homebrew model is the easy case: it already speaks Sleeper ids and you control its
scoring, so it drops straight into the consensus with no mapping at all.

**Migration path:** once this looks right here, it becomes `scripts/build_projections.py`
in the daily job, and the dashboard reads `projections_all.parquet` -- letting you pick
a source or view them side by side. The simulator keeps working unchanged; it just
points at whichever source you choose as its mean.